In [ ]:
from pathlib import Path
from time import perf_counter
import json

import joblib
import optuna
import pandas as pd
from optuna.distributions import (
    CategoricalDistribution,
    FloatDistribution,
    IntDistribution,
)
from optuna.samplers import TPESampler
from optuna_integration.sklearn import OptunaSearchCV

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, make_scorer, precision_score, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

from src.utils.features import split_features

In [ ]:
DATA_DIR = Path("../datasets/final/ml")
SEARCH_METHOD = "optuna"
MODEL_DIR = Path("../models") / SEARCH_METHOD
RESULTS_DIR = Path("../results/model_training") / SEARCH_METHOD

MODEL_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 1
N_SPLITS = 5
N_TRIALS = 30
# trials sequenciais garantem que o TPE use todo o histórico anterior e seja reproduzível.
SEARCH_N_JOBS = 1
#o paralelismo fica dentro dos estimadores que oferecem suporte.
ESTIMATOR_N_JOBS = -1

TRAINING_CONFIGS = {
    "cicids2017": ["cicids2017"],
    "unsw_nb15": ["unsw_nb15"],
    "iot23": ["iot23"],
    "cicids2017_unsw_nb15": ["cicids2017", "unsw_nb15"],
    "cicids2017_iot23": ["cicids2017", "iot23"],
    "unsw_nb15_iot23": ["unsw_nb15", "iot23"],
}

pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", None)

In [ ]:
reference_df = pd.read_parquet(DATA_DIR / "cicids2017_train.parquet")
numerical_features, categorical_features = split_features(reference_df)
feature_columns = numerical_features + categorical_features
del reference_df

def build_pipeline(estimator):
    preprocessor = ColumnTransformer(
        transformers=[
            ("numerical", StandardScaler(), numerical_features),
            ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ]
    )
    return Pipeline(steps=[("preprocessor", preprocessor), ("classifier", estimator)])

In [ ]:
def categorical(values):
    return CategoricalDistribution(values)


MODEL_CONFIGS = {
    "decision_tree": {
        "estimator": DecisionTreeClassifier(random_state=RANDOM_STATE),
        "param_distributions": {
            "classifier__max_depth": categorical([None, 3, 5, 8, 10, 15, 20, 30]),
            "classifier__min_samples_split": IntDistribution(2, 20),
            "classifier__min_samples_leaf": IntDistribution(1, 10),
            "classifier__class_weight": categorical([None, "balanced"]),
        },
    },
    "random_forest": {
        "estimator": RandomForestClassifier(
            random_state=RANDOM_STATE, n_jobs=ESTIMATOR_N_JOBS
        ),
        "param_distributions": {
            "classifier__n_estimators": IntDistribution(100, 300),
            "classifier__max_depth": categorical([None, 10, 15, 20, 30]),
            "classifier__min_samples_leaf": IntDistribution(1, 10),
            "classifier__max_features": categorical(["sqrt", "log2", 0.5, 1.0]),
            "classifier__class_weight": categorical(
                [None, "balanced", "balanced_subsample"]
            ),
        },
    },
    "xgboost": {
        "estimator": XGBClassifier(
            random_state=RANDOM_STATE,
            n_jobs=ESTIMATOR_N_JOBS,
            eval_metric="logloss",
            tree_method="hist",
        ),
        "param_distributions": {
            "classifier__n_estimators": IntDistribution(100, 300),
            "classifier__max_depth": IntDistribution(3, 8),
            "classifier__learning_rate": FloatDistribution(0.03, 0.3, log=True),
            "classifier__subsample": FloatDistribution(0.7, 1.0),
            "classifier__colsample_bytree": FloatDistribution(0.7, 1.0),
        },
    },
    "linear_svm": {
        "estimator": LinearSVC(random_state=RANDOM_STATE),
        "param_distributions": {
            "classifier__C": FloatDistribution(1e-3, 1e2, log=True),
            "classifier__class_weight": categorical([None, "balanced"]),
        },
    },
    "mlp": {
        "estimator": MLPClassifier(random_state=RANDOM_STATE),
        "param_distributions": {
            "classifier__hidden_layer_sizes": categorical([(50,), (100,), (50, 50), (100, 50)]),
            "classifier__alpha": FloatDistribution(1e-5, 1e-2, log=True),
            "classifier__learning_rate_init": FloatDistribution(1e-4, 1e-2, log=True),
            "classifier__early_stopping": categorical([True, False]),
        },
    },
}

In [ ]:
#modelos com orçamento de 30 trials.
MODELS_TO_RUN = [
    "decision_tree",
    "random_forest",
    "xgboost",
    "linear_svm",
    "mlp",
]

cross_validation = StratifiedKFold(
    n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE
)

scoring = {
    "precision_attack": make_scorer(precision_score, pos_label=1, zero_division=0),
    "recall_attack": make_scorer(recall_score, pos_label=1, zero_division=0),
    "f1_attack": make_scorer(f1_score, pos_label=1, zero_division=0),
    "precision_benign": make_scorer(precision_score, pos_label=0, zero_division=0),
    "recall_benign": make_scorer(recall_score, pos_label=0, zero_division=0),
    "f1_benign": make_scorer(f1_score, pos_label=0, zero_division=0),
    "f1_macro": make_scorer(
        f1_score, average="macro", labels=[0, 1], zero_division=0
    ),
    "accuracy": "accuracy",
    "balanced_accuracy": "balanced_accuracy",
}

In [ ]:
def load_training_data(dataset_names):
    columns = feature_columns + ["label_binary", "dataset_name"]
    dataframes = [
        pd.read_parquet(DATA_DIR / f"{dataset_name}_train.parquet", columns=columns)
        for dataset_name in dataset_names
    ]
    training_df = pd.concat(dataframes, ignore_index=True)
    features = training_df[feature_columns]
    target = training_df["label_binary"].eq("ATTACK").astype("int8")
    stratification_labels = (
        training_df["dataset_name"].astype("string")
        + "__"
        + target.astype("string")
    )
    cv_splits = list(cross_validation.split(features, stratification_labels))
    return features, target, cv_splits


def serialize_params(params):
    return json.dumps(params, sort_keys=True, default=lambda value: value.item())

In [ ]:
def train_and_select_model(model_name, training_name, dataset_names):
    features, target, cv_splits = load_training_data(dataset_names)
    model_config = MODEL_CONFIGS[model_name]
    model_dir = MODEL_DIR / model_name
    results_dir = RESULTS_DIR / model_name
    model_dir.mkdir(parents=True, exist_ok=True)
    results_dir.mkdir(parents=True, exist_ok=True)

    study = optuna.create_study(
        direction="maximize",
        sampler=TPESampler(seed=RANDOM_STATE),
        study_name=f"{model_name}_{training_name}",
    )
    search = OptunaSearchCV(
        estimator=build_pipeline(model_config["estimator"]),
        param_distributions=model_config["param_distributions"],
        cv=cv_splits,
        scoring=scoring["f1_macro"],
        n_trials=N_TRIALS,
        study=study,
        refit=True,
        n_jobs=SEARCH_N_JOBS,
        random_state=RANDOM_STATE,
        verbose=1,
        error_score="raise",
        return_train_score=False,
    )

    search_start = perf_counter()
    search.fit(features, target)
    search_elapsed_seconds = perf_counter() - search_start

    candidate_results = pd.DataFrame(search.cv_results_)
    candidate_results = candidate_results.rename(columns={
        **{f"split{fold}_test_score": f"split{fold}_test_f1_macro" for fold in range(N_SPLITS)},
        "mean_test_score": "mean_test_f1_macro",
        "std_test_score": "std_test_f1_macro",
    })
    completed_trials = sorted(
        (trial for trial in study.trials if trial.state.name == "COMPLETE"),
        key=lambda trial: trial.number,
    )
    if len(completed_trials) != len(candidate_results):
        raise RuntimeError("Quantidade de trials concluídos incompatível com cv_results_.")
    candidate_results.insert(0, "trial_number", [trial.number for trial in completed_trials])
    candidate_results.insert(1, "trial_state", [trial.state.name for trial in completed_trials])
    candidate_results.insert(2, "trial_duration_seconds", [
        trial.duration.total_seconds() if trial.duration is not None else None
        for trial in completed_trials
    ])
    candidate_results.insert(3, "params", [
        serialize_params(trial.params) for trial in completed_trials
    ])
    candidate_results["rank_test_f1_macro"] = (
        candidate_results["mean_test_f1_macro"]
        .rank(method="min", ascending=False)
        .astype("int64")
    )
    candidate_results = candidate_results.sort_values("rank_test_f1_macro")
    candidate_results.insert(0, "random_state", RANDOM_STATE)
    candidate_results.insert(0, "n_splits", N_SPLITS)
    candidate_results.insert(0, "n_trials", N_TRIALS)
    candidate_results.insert(0, "training_rows", len(features))
    candidate_results.insert(0, "training_source", training_name)
    candidate_results.insert(0, "search_method", SEARCH_METHOD)
    candidate_results.insert(0, "model", model_name)
    candidate_results.to_csv(results_dir / f"{training_name}_trials.csv", index=False)

    # validação cruzada adicional da configuração vencedora para calcular,
    # em uma única execução, as médias e os desvios-padrão de todas as métricas,
    # pois o OptunaSearchCV aceita apenas uma métrica de scoring por busca.
    winner_cv_n_jobs = (
        1 if model_name in {"random_forest", "xgboost"} else -1
    )
    winner_cv_start = perf_counter()
    winner_cv_results = cross_validate(
        clone(search.best_estimator_),
        features,
        target,
        scoring=scoring,
        cv=cv_splits,
        n_jobs=winner_cv_n_jobs,
        return_train_score=False,
        error_score="raise",
    )
    winner_cv_elapsed_seconds = perf_counter() - winner_cv_start

    fold_results = pd.DataFrame({
        "model": model_name,
        "search_method": SEARCH_METHOD,
        "training_source": training_name,
        "training_rows": len(features),
        "n_trials": N_TRIALS,
        "n_splits": N_SPLITS,
        "random_state": RANDOM_STATE,
        "fold": range(1, N_SPLITS + 1),
        "fit_time_seconds": winner_cv_results["fit_time"],
        "score_time_seconds": winner_cv_results["score_time"],
    })
    for metric in scoring:
        fold_results[metric] = winner_cv_results[f"test_{metric}"]
    fold_results.to_csv(results_dir / f"{training_name}_winner_folds.csv", index=False)

    model_path = model_dir / f"{training_name}.joblib"
    joblib.dump(search.best_estimator_, model_path, compress=3)

    total_elapsed_seconds = search_elapsed_seconds + winner_cv_elapsed_seconds
    result = {
        "model": model_name,
        "search_method": SEARCH_METHOD,
        "training_source": training_name,
        "training_rows": len(features),
        "n_trials": N_TRIALS,
        "n_splits": N_SPLITS,
        "random_state": RANDOM_STATE,
        "selection_metric": "f1_macro",
        "n_completed_trials": len(completed_trials),
        "params": serialize_params(search.best_params_),
        "best_search_f1_macro": search.best_score_,
        "search_elapsed_seconds": search_elapsed_seconds,
        "search_elapsed_minutes": search_elapsed_seconds / 60,
        "refit_time_seconds": search.refit_time_,
        "winner_cv_elapsed_seconds": winner_cv_elapsed_seconds,
        "winner_cv_elapsed_minutes": winner_cv_elapsed_seconds / 60,
        "elapsed_seconds": total_elapsed_seconds,
        "elapsed_minutes": total_elapsed_seconds / 60,
        "fit_time_mean": fold_results["fit_time_seconds"].mean(),
        "fit_time_std": fold_results["fit_time_seconds"].std(ddof=0),
        "score_time_mean": fold_results["score_time_seconds"].mean(),
        "score_time_std": fold_results["score_time_seconds"].std(ddof=0),
        "model_size_mb": model_path.stat().st_size / 1024 ** 2,
        "model_file": model_path.name,
    }
    for metric in scoring:
        result[f"{metric}_mean"] = fold_results[metric].mean()
        result[f"{metric}_std"] = fold_results[metric].std(ddof=0)

    display(candidate_results)
    display(fold_results)
    return result

In [ ]:
all_results = []

for model_name in MODELS_TO_RUN:
    model_results = []
    for training_name, dataset_names in TRAINING_CONFIGS.items():
        print(f"Modelo: {model_name} | origem do treinamento: {training_name}")
        result = train_and_select_model(model_name, training_name, dataset_names)
        model_results.append(result)
        all_results.append(result)

    pd.DataFrame(model_results).to_csv(
        RESULTS_DIR / model_name / "summary.csv", index=False
    )

new_results_df = pd.DataFrame(all_results)
summary_path = RESULTS_DIR / "summary.csv"
if summary_path.exists():
    previous_results_df = pd.read_csv(summary_path).reindex(
        columns=new_results_df.columns
    )
    all_results_df = pd.concat(
        [previous_results_df, new_results_df], ignore_index=True
    ).drop_duplicates(subset=["model", "training_source"], keep="last")
else:
    all_results_df = new_results_df
all_results_df.to_csv(summary_path, index=False)
display(all_results_df)